<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/sankey1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libaries

In [1]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors

In [2]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [3]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [4]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tbl7OYOXduME11uh7")   # Slut-targets

In [5]:
# Explode and rename
t0 = tabel0.copy().explode("Targets (policy targets)").rename(columns={"Targets (policy targets)": "target_id"})
t1 = tabel1.copy().explode("Target Group").rename(columns={"Target Group": "target_group_id"})
t2 = tabel2.copy().explode("Targets").rename(columns={"Targets": "target_id"})

In [6]:
# Register in DuckDB
duckdb.register("tabel0", t0)
duckdb.register("tabel1_exp", t1)
duckdb.register("tabel2_exp", t2)
duckdb.register("tabel3", tabel3)

In [7]:
# SQL query
query = """
SELECT
    t0."Policy source"       AS policy_source,
    t1."Target name"         AS mid_target,
    t2."Target Group"        AS target_group,
    t3."Target name"         AS final_target
FROM
    tabel0 t0
JOIN
    tabel1_exp t1 ON t0.target_id = t1.id
JOIN
    tabel2_exp t2 ON t1.target_group_id = t2.id
JOIN
    tabel3 t3 ON t2.target_id = t3.id
"""

results = duckdb.sql(query).df()

In [8]:
# Trin 1: forkort og saml labels
results['final_target'] = results['final_target'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 50 else x)
source_labels = results['policy_source']
# Lav mapping fra policy source label til A, B, C ...
policy_source_labels = pd.unique(source_labels)
abc_labels = list(string.ascii_uppercase)[:len(policy_source_labels)]
policy_label_map = dict(zip(policy_source_labels, abc_labels))

# Udskift policy source labels i alle lister (kun i source_labels)
source_labels_abc = source_labels.map(policy_label_map)

# Sæt også forklaring til senere brug under diagrammet
policy_label_explanation = {abc: orig for orig, abc in policy_label_map.items()}

middle_labels = results['target_group']
# Target group mapping: 1, 2, 3, ...
target_group_labels = pd.unique(middle_labels)
number_labels = [str(i+1) for i in range(len(target_group_labels))]
target_group_label_map = dict(zip(target_group_labels, number_labels))
middle_labels_num = middle_labels.map(target_group_label_map)
target_group_label_explanation = {num: orig for orig, num in target_group_label_map.items()}

target_labels = results['final_target']
# all_labels = pd.concat([source_labels_abc, middle_labels, target_labels])
all_labels = pd.concat([source_labels_abc, middle_labels_num, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}

In [9]:
# Trin 2: forbindelser
links1 = pd.DataFrame({
    'source': source_labels_abc.map(label_to_index),
    'target': middle_labels_num.map(label_to_index),
    'value': 1
})
links2 = pd.DataFrame({
    'source': middle_labels_num.map(label_to_index),
    'target': target_labels.map(label_to_index),
    'value': 1
})

all_links = pd.concat([links1, links2])

In [10]:
# Brug f.eks. Plotlys palette
node_colors = plotly.colors.qualitative.Plotly

# Tildel farver til unikke labels (noder)
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}

# Liste med farver i samme rækkefølge som unique_labels
node_colors_list = [color_map[label] for label in unique_labels]


# Funktion til at lysne farver (uden brug af gennemsigtighed)
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lysnet farve til hver link ud fra source-node
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in all_links['source']]



# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors
    )
)])
fig.update_layout(title_text="Policy Source → Target Group → Target", font_size=12, height=6000)
fig.show()

print("Forklaring på labels i Sankey-diagrammet:\n")

print("Policy Source (A, B, C...):")
for abc, orig in policy_label_explanation.items():
    print(f"  {abc}: {orig}")

print("\nTarget Group (1, 2, 3...):")
for num, orig in target_group_label_explanation.items():
    print(f"  {num}: {orig}")

Forklaring på labels i Sankey-diagrammet:

Policy Source (A, B, C...):
  A: EU biodiversity strategy 2030
  B: Danmarks fremtidige arealanvendelse
  C: Common Agricultural Policy - Strategic plan 2023-2027
  D: Vandområdeplaner 2021-2027
  E: Mere, bedre og større natur i Danmark
  F: Aftale om et grønt Danmark (grøn trepart)
  G: Danmarks Arealer - Danmarks Fremtid
  H: Forvaltning af Fremtidens Drikkevandsressource
  I: European Green Deal
  J: DK 2030 - Et grønnere, sikrere og stærkere Danmark
  K: Fremskrivning af Råstofforbruget 2022-2040
  L: Notat: fordeling mellem eksiterende og planlagte beskyttede områder i Danmark

Target Group (1, 2, 3...):
  1: Biodiversity
  2: Fisk og fiskerierhverv
  3: Sea and coasts
  4: Groundwater and water
  5: Climate
  6: Sundhed
  7: Environment, pollution and ecotoxins
  8: Agriculture
  9: Fødevaresikkerhed
  10: Økonomi og regional udvikling
  11: Mere natur
  12: Friluftsinteresser
  13: Sårbar natur
  14: Bosætning og infrastruktur
  15: Be

In [11]:
#fig.update_layout(title_text="Policy Source → Target Group → Target", font_size=12, height=2000)
#fig.write_html("sankey_diagram.html")

#from google.colab import files
#files.download("sankey_diagram.html")

Selektering af policy til taget group

In [12]:
# Kun Sankey fra policy source til target group (links1)
# (Du skal have kørt dine celler med source_labels_abc, middle_labels_num og label_to_index først)

# Brug kun links1
links_policy_to_group = links1.copy()

# Saml labels for policy source og target group
labels_policy_to_group = list(source_labels_abc) + list(middle_labels_num)

# Tilpas evt. node-farver hvis du har dem (ellers fjern color-parameter)
# node_colors_policy_to_group = node_colors[:len(labels_policy_to_group)]

import plotly.graph_objects as go

fig_policy_to_group = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="gray", width=0.5),
        label=labels_policy_to_group,
        # color=node_colors_policy_to_group,  # tilføj denne linje hvis du bruger farver
    ),
    link=dict(
        source=links_policy_to_group['source'],
        target=links_policy_to_group['target'],
        value=links_policy_to_group['value'],
        # color=links_policy_to_group['color'],  # tilføj hvis du har farver på links
    )
)])

fig_policy_to_group.update_layout(title_text="Policy Source → Target Group", font_size=12)
fig_policy_to_group.show()
